In [ ]:
%%writefile train.py
import os
import shutil
import sys
import glob
import subprocess

# ==========================================
# 0. EXORCISE THE CACHE (MUST BE FIRST)
# ==========================================

# This guarantees no "en-in" ghost files survive from previous runs
for path in ["/kaggle/working/phoneme_cache", "/kaggle/working/output"]:
    if os.path.exists(path):
        print(f"🧹 Deleting old ghost cache at {path}...")
        shutil.rmtree(path)

# ==========================================
# 1. SETUP & DEPENDENCIES
# ==========================================

try:
    print("⚙️ [1/5] Checking Dependencies...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "numpy>=1.26.0,<2.0",
        "scipy<1.13",
        "transformers<4.43.0",
        "coqui-tts",
        "pysbd",
        "torchaudio==2.2.2"
    ])
    os.system("apt-get update -y && apt-get install -y espeak-ng libsndfile1-dev")
except Exception as e:
    print(f"❌ Setup Failed: {e}")
    sys.exit(1)

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torchaudio")

from trainer import Trainer, TrainerArgs
from TTS.tts.configs.shared_configs import BaseDatasetConfig, BaseAudioConfig
from TTS.tts.configs.vits_config import VitsConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.vits import Vits
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from TTS.utils.audio import AudioProcessor

# ==========================================
# 2. DATASET & FORMATTER
# ==========================================

def find_dataset():
    meta_files = glob.glob("/kaggle/input/**/txt.done.data", recursive=True)

    if not meta_files:
        meta_files = glob.glob("**/txt.done.data", recursive=True)

    if not meta_files:
        print("❌ ERROR: Could not find 'txt.done.data'.")
        sys.exit(1)

    meta_path = meta_files[0]
    root_path = os.path.dirname(meta_path)
    print(f"✅ Found dataset at: {root_path}")

    return root_path, meta_path


ROOT_PATH, META_FILE = find_dataset()

def indic_formatter(root_path, meta_file, **kwargs):
    items = []

    with open(meta_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            content = line.strip("()").strip()
            parts = content.split(" ", 1)

            if len(parts) < 2:
                continue

            file_id = parts[0]
            text = parts[1].replace('"', '').strip()
            wav_file = os.path.join(root_path, "wav", f"{file_id}.wav")

            if os.path.exists(wav_file):
                items.append({
                    "text": text,
                    "audio_file": wav_file,
                    "speaker_name": "ljspeech",
                    "language": "en-us",
                    "root_path": root_path
                })

    return items


dataset_config = BaseDatasetConfig(
    formatter="indic_formatter",
    meta_file_train=META_FILE,
    path=ROOT_PATH,
    language="en-us"
)

# ==========================================
# 3. CONFIGURATION
# ==========================================

config = VitsConfig(
    run_name="vits_hindi_female_resampled",
    batch_size=6,
    eval_batch_size=3,
    batch_group_size=2,
    num_loader_workers=2,
    mixed_precision=True,
    epochs=1000,
    save_step=500,
    output_path="/kaggle/working/output",
    datasets=[dataset_config],
    
    # PHONEMES
    use_phonemes=True,
    phonemizer="espeak",
    phoneme_language="en-us",
    compute_input_seq_cache=True,
    phoneme_cache_path="/kaggle/working/phoneme_cache",
    text_cleaner="phoneme_cleaners",

    # AUDIO
    audio=BaseAudioConfig(
        sample_rate=22050,
        resample=True,
        win_length=1024,
        hop_length=256,
        num_mels=80,
        mel_fmin=0,
        mel_fmax=None
    ),

    use_speaker_embedding=False
)

# ==========================================
# 4. TRAINING LOOP
# ==========================================

if __name__ == "__main__":

    os.makedirs("/kaggle/working/phoneme_cache", exist_ok=True)

    print(f"🔍 DEBUG: Config phoneme_language is: '{config.phoneme_language}'")
    print(f"🔍 DEBUG: Dataset language is: '{dataset_config.language}'")

    # A. Init Processor
    ap = AudioProcessor.init_from_config(config)
    tokenizer, config = TTSTokenizer.init_from_config(config)

    # B. Load Data
    train_samples, eval_samples = load_tts_samples(
        dataset_config,
        eval_split=True,
        formatter=indic_formatter
    )

    # ==========================================
    # C. LOAD CHECKPOINT
    # ==========================================
    
    src = "/kaggle/input/datasets/projectdvl3ystg/hindi-female-checkpoint/hindi_female_checkpoint/checkpoint_1058500.pth"
    dst = "/kaggle/working/checkpoint_1038000.pth"

    if not os.path.exists(dst):
        print("Copying checkpoint to working directory...")
        shutil.copy(src, dst)

    restore_path = dst

    print("✅ Loading checkpoint:")
    print(restore_path)

    # ==========================================
    # D. START TRAINING
    # ==========================================

    print("🔥 Starting Training...")

    model = Vits(config, ap, tokenizer, speaker_manager=None)

    trainer = Trainer(
        TrainerArgs(restore_path=restore_path),
        config,
        output_path="/kaggle/working/output",
        model=model,
        train_samples=train_samples,
        eval_samples=eval_samples,
    )

    trainer.fit()


In [ ]:
!cp -r /kaggle/input/datasets/projectdvl3ystg/hindi-female-checkpoint* /kaggle/working/

!cp -r /kaggle/input/datasets/yashsamant25/hindi-female/* /kaggle/working/
!cd /kaggle/working

!python train.py
